In [56]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## 1. CONFIGURATION & REPRODUCIBILITY


In [57]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DEV_MODE = False
REFERENCE_DATE = datetime(2026, 8, 24)
BASE_START_DATE = datetime(2021, 4, 1)
TARGET_TOTAL_RECORDS = 10000
DUPLICATE_PAIRS = 50

WORK_CONFIG = {
    'road': {'cost_log': 15.5, 'sigma': 0.8, 'days': (180, 450), 'actions': ['Construction of', 'Repair of', 'Widening of']},
    'community_hall': {'cost_log': 15.2, 'sigma': 0.6, 'days': (120, 365), 'actions': ['Construction of', 'Upgradation of']},
    'water_supply': {'cost_log': 14.8, 'sigma': 0.7, 'days': (90, 200), 'actions': ['Installation of', 'Laying pipelines for']},
    'school': {'cost_log': 15.0, 'sigma': 0.7, 'days': (150, 400), 'actions': ['Additional classrooms for', 'Boundary wall for']},
    'drainage': {'cost_log': 14.5, 'sigma': 0.6, 'days': (60, 180), 'actions': ['Construction of drain at', 'Covering of drainage in']}
}

SEVERITY_MAP = {'NONE': 0, 'LOW': 1, 'MEDIUM': 2, 'HIGH': 3, 'CRITICAL': 4}

## 2. LOAD & DEDUPLICATE REAL MP DATA

In [58]:
def load_and_prep_mp_data(filepath="Allocated Limit for Honble MPs.csv"):
    try:
        df = pd.read_csv(filepath, encoding="utf-8-sig")
        df.columns = [c.strip() for c in df.columns]
        df = df.rename(columns={
            "State": "state", "Hon'ble Members of Parliaments": "mp_name",
            "Constituency": "constituency", "Allocated AMOUNT ( ₹ )": "allocated_amount"
        })
        df["allocated_amount"] = pd.to_numeric(df["allocated_amount"], errors="coerce")
    except FileNotFoundError:
        if DEV_MODE:
            df = pd.DataFrame({
                "state": ["MAHARASHTRA", "UP", "BIHAR"] * 50,
                "constituency": [f"CONST_{i}" for i in range(150)],
                "mp_name": [f"MP_{i}" for i in range(150)],
                "allocated_amount": [147000000] * 130 + [50000000] * 20
            })
        else:
            raise FileNotFoundError("CRITICAL: Real MPLADS Allocation CSV missing. DEV_MODE is False.")

    df = df.dropna(subset=["allocated_amount", "mp_name"])
    df["state"] = df["state"].str.strip().str.title()
    df["constituency"] = df["constituency"].str.strip().str.upper()
    df = df.drop_duplicates(subset=["mp_name", "constituency"]).reset_index(drop=True)
    df["mp_id"] = ["MPID_" + str(i).zfill(4) for i in range(len(df))]
    return df

## 3. GENERATE HETEROGENEOUS NORMAL PROJECTS

In [59]:
def generate_normal_population(mp_df, n_target):
    raw_weights = np.clip(np.random.pareto(a=4.0, size=len(mp_df)), 0, 4.0) + 1
    proportions = raw_weights / raw_weights.sum()
    counts = np.floor(proportions * n_target).astype(int)

    remainder = n_target - counts.sum()
    add_idx = np.random.choice(len(mp_df), size=remainder, replace=True)
    for idx in add_idx:
        counts[idx] += 1

    records = []
    project_counter = 1

    for mp_idx, num_projects in enumerate(counts):
        if num_projects <= 0: continue
        mp = mp_df.iloc[mp_idx]

        target_total_sanction = mp['allocated_amount'] * np.random.uniform(0.70, 0.99)
        w_types = np.random.choice(list(WORK_CONFIG.keys()), num_projects)
        raw_costs = [np.random.lognormal(WORK_CONFIG[wt]['cost_log'], WORK_CONFIG[wt]['sigma']) for wt in w_types]
        scale_factor = target_total_sanction / sum(raw_costs)

        for i in range(num_projects):
            wt = w_types[i]
            sanctioned = round(raw_costs[i] * scale_factor, 2)
            estimated = round(sanctioned * np.random.uniform(1.0, 1.15), 2)

            loc_id = f"LOC_{mp['mp_id']}_{random.randint(1, 100)}"
            action = random.choice(WORK_CONFIG[wt]['actions'])
            desc = f"{action} {wt.replace('_', ' ')} at {loc_id}"

            expected_dur = random.randint(WORK_CONFIG[wt]['days'][0], WORK_CONFIG[wt]['days'][1])

            lifecycle_choice = np.random.choice(['completed', 'in_progress', 'sanctioned'], p=[0.60, 0.30, 0.10])

            if lifecycle_choice == 'completed':
                start_date = REFERENCE_DATE - timedelta(days=random.randint(expected_dur, expected_dur + 800))
            elif lifecycle_choice == 'in_progress':
                start_date = REFERENCE_DATE - timedelta(days=random.randint(30, max(31, expected_dur - 5)))
            else:
                start_date = REFERENCE_DATE - timedelta(days=random.randint(0, 29))

            expected_completion = start_date + timedelta(days=expected_dur)
            project_age = (REFERENCE_DATE - start_date).days

            if project_age < 30:
                status, phys_prog, fin_prog, actual_completion = "sanctioned", 0.0, 0.0, pd.NaT
            else:
                base_prog = (project_age / expected_dur) * 100
                phys_prog = min(100.0, max(0.0, np.random.normal(base_prog, 15.0)))
                fin_prog = min(100.0, max(0.0, np.random.normal(phys_prog, 10.0)))

                if lifecycle_choice == 'completed':
                    status = "completed"
                    phys_prog, fin_prog = 100.0, np.random.uniform(95.0, 100.0)
                    actual_completion = min(expected_completion + timedelta(days=random.randint(-20, 60)), REFERENCE_DATE)
                else:
                    status, actual_completion = "in_progress", pd.NaT

            expenditure = round(sanctioned * (fin_prog / 100.0), 2)
            payment_count = int(fin_prog / np.random.uniform(15, 30)) + (1 if fin_prog > 0 else 0)

            records.append({
                'project_id': f"PRJ_{project_counter:06d}", 'mp_id': mp['mp_id'], 'state': mp['state'],
                'constituency': mp['constituency'], 'allocated_amount': mp['allocated_amount'],
                'work_type': wt, 'location_id': loc_id, 'project_description': desc,
                'estimated_cost': estimated, 'sanctioned_amount': sanctioned,
                'baseline_expenditure': expenditure,
                'expenditure': expenditure,
                'payment_count': payment_count,
                'start_date': start_date, 'expected_completion': expected_completion, 'actual_completion': actual_completion,
                'status': status, 'physical_progress_pct': round(phys_prog, 2), 'financial_progress_pct': round(fin_prog, 2),
                'is_injected_anomaly': 0, 'anomaly_type': "none", 'anomaly_severity': "NONE", 'severity_score': 0,
                'duplicate_group_id': "none"
            })
            project_counter += 1

    return pd.DataFrame(records)

## 4. INJECT CONTROLLED ANOMALIES & FUZZED DUPLICATES

In [60]:
def inject_anomalies_and_duplicates(df):
    df = df.copy()
    n = len(df)

    avail_idx = list(df.index)
    np.random.shuffle(avail_idx)

    dup_src_idx = avail_idx[:DUPLICATE_PAIRS]
    dup_target_idx = avail_idx[DUPLICATE_PAIRS:DUPLICATE_PAIRS*2]
    anomaly_pool = set(avail_idx[DUPLICATE_PAIRS*2:])

    # --- 4A. DUPLICATE WORK ---
    clones = df.loc[dup_src_idx].copy().reset_index(drop=True)
    grp_counter = 1
    for i in range(len(clones)):
        orig_idx = dup_src_idx[i]
        grp_id = f"DUP_GRP_{grp_counter:04d}"

        df.at[orig_idx, 'is_injected_anomaly'] = 1
        df.at[orig_idx, 'anomaly_type'] = 'duplicate_work'
        df.at[orig_idx, 'anomaly_severity'] = 'HIGH'
        df.at[orig_idx, 'duplicate_group_id'] = grp_id

        cost_fuzz = np.random.uniform(0.98, 1.02)
        clones.at[i, 'sanctioned_amount'] = round(clones.at[i, 'sanctioned_amount'] * cost_fuzz, 2)
        clones.at[i, 'estimated_cost'] = round(clones.at[i, 'estimated_cost'] * cost_fuzz, 2)
        clones.at[i, 'baseline_expenditure'] = 0.0
        clones.at[i, 'expenditure'] = round(df.at[orig_idx, 'expenditure'] * cost_fuzz, 2)

        fuzzed_start = clones.at[i, 'start_date'] + timedelta(days=random.randint(1, 4))
        clones.at[i, 'start_date'] = min(fuzzed_start, REFERENCE_DATE)

        clones.at[i, 'project_id'] = f"PRJ_DUP_{grp_counter:06d}"
        clones.at[i, 'is_injected_anomaly'] = 1
        clones.at[i, 'anomaly_type'] = 'duplicate_work'
        clones.at[i, 'anomaly_severity'] = 'HIGH'
        clones.at[i, 'duplicate_group_id'] = grp_id
        grp_counter += 1

    df.iloc[dup_target_idx] = clones.values

    # --- 4B. OTHER ANOMALIES ---
    def get_idx_by_status(count, allowed_statuses):
        valid = [i for i in anomaly_pool if df.at[i, 'status'] in allowed_statuses]
        if len(valid) < count:
            raise ValueError(f"Not enough candidates for statuses {allowed_statuses}")
        selected = valid[:count]
        for i in selected:
            anomaly_pool.remove(i)
        return selected

    counts = {
        'cost_overrun': int(n * 0.03), 'project_delay': int(n * 0.03),
        'low_fund_utilization': int(n * 0.02), 'mismatch_phys_fin': int(n * 0.02),
        'combined': int(n * 0.01)
    }

    for i in get_idx_by_status(counts['cost_overrun'], ['completed', 'in_progress']):
        sev = np.random.choice(['LOW', 'MEDIUM', 'HIGH'], p=[0.5, 0.3, 0.2])
        mult = np.random.uniform(1.05, 1.20) if sev == 'LOW' else np.random.uniform(1.20, 1.60) if sev == 'MEDIUM' else np.random.uniform(1.60, 2.50)
        df.at[i, 'expenditure'] = round(df.at[i, 'sanctioned_amount'] * mult, 2)
        df.at[i, 'financial_progress_pct'] = round((df.at[i, 'expenditure'] / df.at[i, 'sanctioned_amount']) * 100, 2)
        df.at[i, 'is_injected_anomaly'] = 1
        df.at[i, 'anomaly_type'] = 'cost_overrun'
        df.at[i, 'anomaly_severity'] = sev

    for i in get_idx_by_status(counts['project_delay'], ['completed', 'in_progress']):
        wt = df.at[i, 'work_type']
        expected_dur = random.randint(WORK_CONFIG[wt]['days'][0], WORK_CONFIG[wt]['days'][1])
        overdue_days = random.randint(30, 200)
        df.at[i, 'start_date'] = REFERENCE_DATE - timedelta(days=expected_dur + overdue_days)
        df.at[i, 'expected_completion'] = df.at[i, 'start_date'] + timedelta(days=expected_dur)

        df.at[i, 'status'] = 'delayed'
        df.at[i, 'actual_completion'] = pd.NaT
        df.at[i, 'is_injected_anomaly'] = 1
        df.at[i, 'anomaly_type'] = 'project_delay'
        df.at[i, 'anomaly_severity'] = 'MEDIUM'

    for i in get_idx_by_status(counts['low_fund_utilization'], ['in_progress']):
        df.at[i, 'status'] = 'in_progress'
        df.at[i, 'physical_progress_pct'] = round(np.random.uniform(5, 15), 2)
        df.at[i, 'financial_progress_pct'] = round(np.random.uniform(2, 10), 2)
        df.at[i, 'expenditure'] = round(df.at[i, 'sanctioned_amount'] * (df.at[i, 'financial_progress_pct']/100), 2)
        df.at[i, 'is_injected_anomaly'] = 1
        df.at[i, 'anomaly_type'] = 'low_fund_utilization'
        df.at[i, 'anomaly_severity'] = 'MEDIUM'

    for i in get_idx_by_status(counts['mismatch_phys_fin'], ['completed', 'in_progress']):
        df.at[i, 'physical_progress_pct'] = round(np.random.uniform(10, 30), 2)
        df.at[i, 'financial_progress_pct'] = round(np.random.uniform(85, 100), 2)
        df.at[i, 'expenditure'] = round(df.at[i, 'sanctioned_amount'] * (df.at[i, 'financial_progress_pct']/100), 2)
        df.at[i, 'is_injected_anomaly'] = 1
        df.at[i, 'anomaly_type'] = 'financial_physical_mismatch'
        df.at[i, 'anomaly_severity'] = 'HIGH'

    for i in get_idx_by_status(counts['combined'], ['completed', 'in_progress']):
        wt = df.at[i, 'work_type']
        expected_dur = random.randint(WORK_CONFIG[wt]['days'][0], WORK_CONFIG[wt]['days'][1])
        overdue_days = random.randint(30, 200)
        df.at[i, 'start_date'] = REFERENCE_DATE - timedelta(days=expected_dur + overdue_days)
        df.at[i, 'expected_completion'] = df.at[i, 'start_date'] + timedelta(days=expected_dur)

        df.at[i, 'status'] = 'delayed'
        df.at[i, 'actual_completion'] = pd.NaT
        df.at[i, 'expenditure'] = round(df.at[i, 'sanctioned_amount'] * np.random.uniform(1.8, 3.0), 2)
        df.at[i, 'financial_progress_pct'] = round((df.at[i, 'expenditure'] / df.at[i, 'sanctioned_amount']) * 100, 2)
        df.at[i, 'is_injected_anomaly'] = 1
        df.at[i, 'anomaly_type'] = 'combined_anomaly'
        df.at[i, 'anomaly_severity'] = 'CRITICAL'

    non_completed_mask = df['status'].isin(['in_progress', 'sanctioned', 'delayed'])
    df.loc[non_completed_mask, 'actual_completion'] = pd.NaT

    df['project_remaining_balance'] = (df['sanctioned_amount'] - df['expenditure']).round(2)
    df['expenditure_delta'] = (df['expenditure'] - df['baseline_expenditure']).round(2)

    # Map Severity Score
    df['severity_score'] = df['anomaly_severity'].map(SEVERITY_MAP)

    # ENFORCE PAYMENT COUNT CONSISTENCY
    raw_payments = (df['financial_progress_pct'] / np.random.uniform(15, 30, len(df))).fillna(0).astype(int)
    mask_exp_pos = df['expenditure'] > 0
    mask_exp_zero = df['expenditure'] == 0
    df.loc[mask_exp_pos, 'payment_count'] = np.maximum(1, raw_payments[mask_exp_pos])
    df.loc[mask_exp_zero, 'payment_count'] = 0

    return df

## 5. COUNTERFACTUAL CAUSATION & AGGREGATIONS

In [61]:
def calculate_breaches_and_balances(df):
    mp_agg = df.groupby('mp_id').agg(
        actual_total_expenditure=('expenditure', 'sum'),
        baseline_total_expenditure=('baseline_expenditure', 'sum'),
        allocated_amount=('allocated_amount', 'first')
    ).reset_index()

    conditions = [
        (mp_agg['actual_total_expenditure'] > mp_agg['allocated_amount']) & (mp_agg['baseline_total_expenditure'] <= mp_agg['allocated_amount']),
        (mp_agg['actual_total_expenditure'] > mp_agg['allocated_amount']) & (mp_agg['baseline_total_expenditure'] > mp_agg['allocated_amount'])
    ]
    choices = ['ANOMALY_CAUSED_BREACH', 'BASELINE_BREACH']
    mp_agg['mp_allocation_breach_cause'] = np.select(conditions, choices, default='NONE')
    mp_agg['mp_remaining_allocation'] = (mp_agg['allocated_amount'] - mp_agg['actual_total_expenditure']).round(2)

    df = df.merge(mp_agg[['mp_id', 'mp_allocation_breach_cause', 'mp_remaining_allocation']], on='mp_id', how='left')
    return df

## 6. STRICT ASSERTION VALIDATION

In [62]:
def run_v3_2_6_strict_assertions(df):
    # 1. Structural
    assert len(df) == TARGET_TOTAL_RECORDS
    assert df['project_id'].is_unique

    # 2. Vocabulary & Metadata Normalization
    valid_severities = {'NONE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'}
    assert set(df['anomaly_severity'].unique()).issubset(valid_severities), "Invalid severity vocabulary"
    assert (df[df['is_injected_anomaly'] == 0]['severity_score'] == 0).all(), "Normal rows must have severity_score = 0"
    assert (df[df['is_injected_anomaly'] == 1]['severity_score'] >= 1).all(), "Anomaly rows must have severity_score >= 1"
    assert (df['severity_score'] == df['anomaly_severity'].map(SEVERITY_MAP)).all(), "Score mapping mismatch"

    # 3. Blockers
    assert len(df[(df['status'] == 'sanctioned') & (df['expenditure'] > 0)]) == 0
    assert len(df[(df['status'] == 'sanctioned') & (df['payment_count'] > 0)]) == 0
    assert len(df[(df['expenditure'] > 0) & (df['payment_count'] == 0)]) == 0
    assert len(df[df['payment_count'] < 0]) == 0
    assert (df['start_date'] <= REFERENCE_DATE).all()

    # 4. Topology & Anomaly Counts
    dup_rows = df[df['duplicate_group_id'] != 'none']
    assert dup_rows['duplicate_group_id'].nunique() == DUPLICATE_PAIRS
    assert (dup_rows['duplicate_group_id'].value_counts() == 2).all()
    assert len(df[df['anomaly_type'] == 'cost_overrun']) == int(TARGET_TOTAL_RECORDS * 0.03)
    assert len(df[df['anomaly_type'] == 'project_delay']) == int(TARGET_TOTAL_RECORDS * 0.03)
    assert len(df[df['anomaly_type'] == 'low_fund_utilization']) == int(TARGET_TOTAL_RECORDS * 0.02)
    assert len(df[df['anomaly_type'] == 'financial_physical_mismatch']) == int(TARGET_TOTAL_RECORDS * 0.02)
    assert len(df[df['anomaly_type'] == 'duplicate_work']) == DUPLICATE_PAIRS * 2
    assert len(df[df['anomaly_type'] == 'combined_anomaly']) == int(TARGET_TOTAL_RECORDS * 0.01)

## EXECUTION

In [63]:
if __name__ == "__main__":
    mp_lookup = load_and_prep_mp_data()
    base_projects = generate_normal_population(mp_lookup, TARGET_TOTAL_RECORDS)
    final_projects_pre = inject_anomalies_and_duplicates(base_projects)
    final_projects = calculate_breaches_and_balances(final_projects_pre)

    run_v3_2_6_strict_assertions(final_projects)
    final_projects.to_csv("synthetic_projects_raw_v3.2.6.csv", index=False)